# 🧬 Cuaderno 05 - Secuencias y k-mers

**Curso:**: Bioinformática
**Tema:**: Secuencias y k-mers
**Repositorio:**: `bio-notebooks`
**Herramientas:**: Python, Jupyter Notebook, Biopython

## Competencias del curso y del laboratorio
- Implementar algoritmos de alineamiento de secuencias global utilizando programación dinámica.
- Desarrollar algoritmos eficientes para el conteo y análisis de k-mers en secuencias biológicas, identificando patrones y visualizando información genómica.
- Aplicar conceptos fundamentales de la Bioinformática para analizar similitud entre secuencias biológicas y extraer información relevante de datos genómicos.
- Utilizar herramientas y bibliotecas de Python, como Biopython, para manipular y analizar secuencias biológicas, facilitando la interpretación de datos genómicos y la visualización de resultados.

## Contenido
1. Marco teórico
2. Implementación de algoritmos de conteo de k-mers
3. Análisis de frecuencias de k-mers en secuencias biológicas
4. Visualización de resultados y patrones de k-mers
5. Aplicaciones prácticas en genómica y metagenómica

## 1. Marco teórico
En esta sección se introduce el concepto de k-mers, su importancia en el análisis genómico y las aplicaciones prácticas en bioinformática. Se discuten las técnicas de conteo de k-mers, las estructuras de datos utilizadas para su almacenamiento y los desafíos computacionales asociados con el análisis de k-mers en grandes conjuntos de datos genómicos.

Los k-mers son subsecuencias contiguas de longitud k extraídas a partir de una secuencia biológica, ya sea ADN, ARN o proteínas. El análisis de k-mers constituye una de las técnicas fundamentales para el procesamiento y estudio de información genómica debido a su eficiencia computacional y su capacidad para identificar patrones biológicos relevantes.

Dada una secuencia de longitud n, el número total de k-mers que pueden extraerse se calcula mediante:
n − k + 1

Por ejemplo, para la secuencia ATGCA y k = 3, los k-mers obtenidos son:
- ATG
- TGC
- GCA

En secuencias de ADN existen cuatro nucleótidos posibles (A,T,C,G), por lo que el número total de combinaciones posibles de k-mers está dado por 4^k.

El análisis de frecuencias de k-mers permite identificar regiones repetitivas, patrones evolutivos y similitudes entre organismos. Además, constituye la base de múltiples aplicaciones modernas, incluyendo:
- ensamblaje de genomas
- alineamiento de secuencias
- metagenómica
- detección de mutaciones
- clasificación taxonómica
- búsqueda de similitud genética

Desde el punto de vista computacional, el conteo de k-mers requiere algoritmos eficientes debido al gran volumen de datos genómicos. Para ello, suelen emplearse estructuras de datos como tablas hash, diccionarios y árboles Trie, así como técnicas de optimización mediante ventanas deslizantes (sliding window) y paralelización.

En aplicaciones reales, los valores de k suelen variar entre 3 y 31 dependiendo del objetivo del análisis. Valores pequeños permiten detectar patrones generales, mientras que valores grandes ofrecen mayor especificidad genética, aunque incrementan el costo computacional y el uso de memoria.

In [3]:
# Importar bibliotecas necesarias
from Bio import SeqIO
from collections import defaultdict
import matplotlib.pyplot as plt
import os

### Funciones necesarias para el análisis de k-mers

In [15]:
#Leer fasta
def leer_fasta(filepath: str) -> tuple[str, str, str]:
    """Lee un archivo FASTA y nos devuelve el ID, la descripción y la secuencia
    Arguments:
        filepath (str): Es la ruta al archivo FASTA

    Returns:
        tuple[str, str, str]: ID, descripción y secuencia
    """
    with open(filepath, 'r') as file:
        lineas = file.readlines()
    header = lineas[0].strip()
    print("header: ",header)
    print("header.split(): ",header.split())
    secuencia_id = header.split()[0][1:]  # Eliminar el '>' y obtener el ID
    descripcion = ' '.join(header.split()[1:])  # Obtener la descripción
    secuencia = ''.join(linea.strip() for linea in lineas[1:]).upper()  # Unir las líneas de la secuencia
    return secuencia_id, descripcion, secuencia

## 2. Implementación de algoritmos de conteo de k-mers
En esta sección se implementan algoritmos para el conteo de k-mers en secuencias biológicas utilizando Python. Se presentan diferentes enfoques, incluyendo el uso de diccionarios para almacenar las frecuencias de k-mers y técnicas de optimización para mejorar la eficiencia del conteo. Además, se discuten las ventajas y desventajas de cada método, así como su aplicabilidad en diferentes contextos genómicos.

### 2.1. Lectura de secuencias desde archivos FASTA
En esta subsección se muestra cómo leer secuencias biológicas desde archivos FASTA utilizando la biblioteca Biopython. Se implementa una función que extrae el ID, la descripción y la secuencia de un archivo FASTA, facilitando su posterior análisis de k-mers. Además, se discuten las consideraciones importantes al manejar archivos FASTA, como la gestión de secuencias largas y la normalización de caracteres.

In [16]:
DATA_DIR = 'data/'
filepath = DATA_DIR + 'ejemplo_labo.fasta'
secuencia_id, descripcion, secuencia = leer_fasta(filepath=filepath)

#Impresion de resultados
print(f"ID: {secuencia_id}")
print(f"Descripción: {descripcion}")
print(f"Secuencia: {secuencia}")

header:  >ejemplo_labo.3 Mus musculus breast cancer 1, early onset (Brca1), mRNA
header.split():  ['>ejemplo_labo.3', 'Mus', 'musculus', 'breast', 'cancer', '1,', 'early', 'onset', '(Brca1),', 'mRNA']
ID: ejemplo_labo.3
Descripción: Mus musculus breast cancer 1, early onset (Brca1), mRNA
Secuencia: ATGCA
